# 🎥 Person Tracking — Disappearance & Reappearance Counter
This notebook:
- Accepts any video file
- Detects **persons** using YOLOv8/YOLO26
- Assigns a **persistent ID** to each person via ByteTrack
- Counts how many times each person **disappears** and **reappears**
- Saves an annotated output video and prints a final summary table

## 2. Configuration — set your paths here

In [1]:
# ─── USER CONFIG ──────────────────────────────────────────────────────────────
VIDEO_PATH   = "/path/to/your/video.mp4"   # ← change this
MODEL_PATH   = "yolo11n.pt"                # or "yolo26n.pt" if you have it
OUTPUT_PATH  = "tracked_output.mp4"        # annotated video saved here

# A person is considered "disappeared" when they are absent for this many
# consecutive frames. Lower = more sensitive, higher = more lenient.
DISAPPEAR_GRACE_FRAMES = 30   # ~1 sec at 30 fps

CONF_THRESHOLD = 0.4          # minimum detection confidence
# ──────────────────────────────────────────────────────────────────────────────

## 3. Imports

In [2]:
import cv2
from collections import defaultdict
from ultralytics import YOLO
from IPython.display import display
import os

print("All imports OK ✅")

All imports OK ✅


## 4. Load model & set up tracking state

In [ ]:
model = YOLO(MODEL_PATH)

# Class index for 'person' in COCO (used by all YOLO detection models)
PERSON_CLASS_ID = 0

# ── Per-ID tracking state ─────────────────────────────────────────────────────
# last_seen[id]        = frame index when this ID was last detected
# disappear_count[id]  = number of times this ID has disappeared
# reappear_count[id]   = number of times this ID has reappeared
# is_gone[id]          = True while the ID is currently "missing"
last_seen       = {}          # id → last frame index
disappear_count = defaultdict(int)
reappear_count  = defaultdict(int)
is_gone         = defaultdict(bool)   # False = currently visible / just appeared

print("Model loaded ✅")

## 5. Process video — detect, track, count

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Cannot open video: {VIDEO_PATH}"

fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {width}×{height} @ {fps:.1f} fps | {total} frames")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

frame_idx = 0
active_ids_prev = set()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ── Run tracker (ByteTrack is built into Ultralytics) ────────────────────
    results = model.track(
        frame,
        persist=True,            # keeps tracker state across frames
        classes=[PERSON_CLASS_ID],
        conf=CONF_THRESHOLD,
        tracker="bytetrack.yaml",
        verbose=False
    )[0]

    # ── Extract tracked person IDs in this frame ─────────────────────────────
    active_ids_curr = set()

    if results.boxes.id is not None:
        for box, track_id, conf in zip(
            results.boxes.xyxy.cpu().numpy(),
            results.boxes.id.cpu().numpy().astype(int),
            results.boxes.conf.cpu().numpy()
        ):
            tid = int(track_id)
            active_ids_curr.add(tid)
            last_seen[tid] = frame_idx

            # ── Reappearance detection ────────────────────────────────────────
            if is_gone[tid]:
                reappear_count[tid] += 1
                is_gone[tid] = False

            # ── Draw bounding box + ID + stats ───────────────────────────────
            x1, y1, x2, y2 = map(int, box)
            color = (
                (50 + (tid * 77) % 180),
                (80 + (tid * 131) % 150),
                (100 + (tid * 53) % 155),
            )
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            label = (
                f"ID:{tid}  "
                f"gone:{disappear_count[tid]}  "
                f"back:{reappear_count[tid]}"
            )
            (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
            cv2.rectangle(frame, (x1, y1 - lh - 6), (x1 + lw + 4, y1), color, -1)
            cv2.putText(frame, label, (x1 + 2, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    # ── Disappearance detection ───────────────────────────────────────────────
    # Any ID that was active last frame but absent for > grace period → gone
    for tid in list(last_seen.keys()):
        if tid not in active_ids_curr:
            absent_for = frame_idx - last_seen[tid]
            if absent_for == DISAPPEAR_GRACE_FRAMES and not is_gone[tid]:
                disappear_count[tid] += 1
                is_gone[tid] = True

    # ── Frame counter overlay ─────────────────────────────────────────────────
    n_persons = len(active_ids_curr)
    cv2.putText(
        frame,
        f"Frame {frame_idx}/{total}  |  Persons: {n_persons}",
        (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
    )

    writer.write(frame)
    active_ids_prev = active_ids_curr
    frame_idx += 1

    if frame_idx % 100 == 0:
        print(f"  processed {frame_idx}/{total} frames …")

cap.release()
writer.release()
print(f"\n✅ Done! Annotated video saved to: {OUTPUT_PATH}")

## 6. Summary report

In [ ]:
all_ids = sorted(set(list(disappear_count.keys()) + list(reappear_count.keys()) + list(last_seen.keys())))

rows = []
for tid in all_ids:
    d = disappear_count[tid]
    r = reappear_count[tid]
    rows.append({
        "Person ID":          tid,
        "Times Disappeared":  d,
        "Times Reappeared":   r,
        "Still in frame?":    "Yes" if not is_gone[tid] else "No",
    })

df = pd.DataFrame(rows)

print("═" * 55)
print(f"  Total unique persons tracked : {len(all_ids)}")
print(f"  Total video frames processed : {frame_idx}")
print(f"  Video FPS                    : {fps:.1f}")
print("═" * 55)
print()
display(df.to_string(index=False))

## 7. Bar chart — disappearances vs reappearances per person

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if df.empty:
    print("No persons tracked.")
else:
    x      = np.arange(len(df))
    width  = 0.35
    labels = [f"ID {pid}" for pid in df["Person ID"]]

    fig, ax = plt.subplots(figsize=(max(6, len(df) * 1.2), 5))
    bars1 = ax.bar(x - width/2, df["Times Disappeared"], width, label="Disappeared", color="#e74c3c")
    bars2 = ax.bar(x + width/2, df["Times Reappeared"],  width, label="Reappeared",  color="#2ecc71")

    ax.set_xlabel("Person ID")
    ax.set_ylabel("Count")
    ax.set_title("Per-Person Disappearance & Reappearance Counts")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.bar_label(bars1, padding=3)
    ax.bar_label(bars2, padding=3)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.savefig("tracking_summary.png", dpi=150)
    plt.show()
    print("Chart saved to tracking_summary.png")

## 8. Export summary to CSV

In [ ]:
csv_path = "tracking_summary.csv"
df.to_csv(csv_path, index=False)
print(f"Summary exported to {csv_path}")